# Notebook 02c: Forge → ESMFold → ProteinMPNN → Protenix → Filter

## 2 × 3 Experimental Design — Forge conditions

| `TAU_STATE` | Target Sequence | Description |
|-------------|----------------|-------------|
| `monomer`  | Full K18 (168 aa) | Disordered monomer, all epitopes exposed |
| `oligomer` | PHF6*+PHF6 tandem repeat (79 aa) | Mimics stacked oligomeric interface |
| `fibril`   | PHF6*/PHF6 steric zipper (38 aa) | Aggregation-prone fibril-forming region |

**Run once per `TAU_STATE`** by changing the config cell and re-running.  
Outputs are tagged `forge_{tau_state}_top{N}.csv` then upload all 3 to NB03.

```
K18 target sequence (selected by TAU_STATE)
    ↓
Forge (InferenceWrapper, uv venv)  →  forge_sequences.csv
    ↓
ESMFold pass 1  →  pLDDT filter (≥70)  →  backbones
    ↓
ProteinMPNN (monomer)  →  4 redesigned sequences per backbone
    ↓  free GPU
Protenix (binder + target)  →  ipTM + ranking_score
    ↓
Top-N by ranking_score  →  forge_{tau_state}_top{N}.csv  <- feed into NB03
```

**Requires**: GPU (T4, ≥16 GB)  |  **Runtime**: ~2–2.5 h


## 0. GPU check

In [71]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU — switch to GPU runtime')

Mon Mar 16 07:12:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [72]:
# ── Google Drive Setup ────────────────────────────────────────────────────────
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/CHEM_280'
    _in_drive  = True
    print(f'Drive mounted. Base: {DRIVE_BASE}')
except Exception:
    DRIVE_BASE = '/content'
    _in_drive  = False
    print('Drive not available — saving to /content/')

NB02C_OUT = os.path.join(DRIVE_BASE, 'results', 'nb02c_output')
os.makedirs(NB02C_OUT, exist_ok=True)
print(f'NB02c output dir: {NB02C_OUT}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Base: /content/drive/MyDrive/CHEM_280
NB02c output dir: /content/drive/MyDrive/CHEM_280/results/nb02c_output


## 1. Install

In [73]:
import os
import sys
import subprocess
import re

# Remove existing ProteinMPNN directory before cloning
!rm -rf /content/ProteinMPNN
!git clone https://github.com/dauparas/ProteinMPNN /content/ProteinMPNN

# Install other dependencies
!pip install fair-esm biopython pandas matplotlib

# Correctly install Protenix from the official ByteDance repository
print("Installing official Protenix from GitHub...")
!rm -rf /content/Protenix
!git clone https://github.com/bytedance/Protenix.git /content/Protenix
!pip install -e /content/Protenix


Cloning into '/content/ProteinMPNN'...
remote: Enumerating objects: 634, done.
remote: Counting objects: 100% (271/271), done.
remote: Compressing objects: 100% (125/125), done.
remote: Total 634 (delta 151), reused 146 (delta 146), pack-reused 363 (from 1)
Receiving objects: 100% (634/634), 119.90 MiB | 39.67 MiB/s, done.
Resolving deltas: 100% (290/290), done.
Installing official Protenix from GitHub...
Cloning into '/content/Protenix'...
remote: Enumerating objects: 2469, done.
remote: Counting objects: 100% (1193/1193), done.
remote: Compressing objects: 100% (494/494), done.
remote: Total 2469 (delta 764), reused 705 (delta 698), pack-reused 1276 (from 2)
Receiving objects: 100% (2469/2469), 95.40 MiB | 42.49 MiB/s, done.
Resolving deltas: 100% (1530/1530), done.
Obtaining file:///content/Protenix
  Preparing metadata (setup.py) ... done
  Running setup.py develop for protenix


In [ ]:
import sys, os, subprocess



import esm

# Check Protenix CLI
r = subprocess.run(['protenix', '--help'], capture_output=True, text=True)
protenix_ok = r.returncode == 0 or 'usage' in r.stdout.lower() or 'usage' in r.stderr.lower()

print('ESMFold  :', 'OK')
print('ProteinMPNN:', 'OK' if os.path.exists('/content/ProteinMPNN/protein_mpnn_run.py') else 'MISSING')
print('Protenix :', 'OK' if protenix_ok else 'check install')

## 2. Configuration

In [ ]:
import os

# ── TAU STATE SELECTOR ────────────────────────────────────────────────────────
# Change this to run a different condition. Options: 'monomer', 'oligomer', 'fibril'
TAU_STATE = 'monomer'

# ── Target sequences ──────────────────────────────────────────────────────────
TARGET_SEQUENCE_MONOMER = (
    'KVQIINKKLDLSNVQSKCGSKDNIKHVPGGGSVQIVYKPVDLSKVTSKCGSLGNIHHKPGGG'
    'QVEVKSEKLDFKDRVQSKIGSLDNITHVPGGGNKKIETHKLTFRENAKAKTDHGAEIVYKSPV'
    'VSGDTSPRHLSNVSSTGSIDMVDSPQLATLADEVSASLAKQGL'
)
TARGET_SEQUENCE_OLIGOMER = (
    'KVQIINKKLDLSNVQSKCGSKDNIKHVPGGGSVQIVYK'
    'GGG'
    'KVQIINKKLDLSNVQSKCGSKDNIKHVPGGGSVQIVYK'
)  # 79 aa
TARGET_SEQUENCE_FIBRIL = 'KVQIINKKLDLSNVQSKCGSKDNIKHVPGGGSVQIVYK'  # 38 aa

_TARGET_MAP = {
    'monomer':  TARGET_SEQUENCE_MONOMER,
    'oligomer': TARGET_SEQUENCE_OLIGOMER,
    'fibril':   TARGET_SEQUENCE_FIBRIL,
}
if TAU_STATE not in _TARGET_MAP:
    raise ValueError(f'Unknown TAU_STATE: {TAU_STATE!r}. Options: monomer, oligomer, fibril')
TARGET_SEQUENCE = _TARGET_MAP[TAU_STATE]

# ── Epitope ranges (0-indexed on K18 monomer) ────────────────────────────────
EPITOPE_RANGES = [
    (1,   7,  'PHF6star_VQIINK', 1.0),
    (32,  38, 'PHF6_VQIVYK',     1.0),
    (51,  70, 'jR2R3',           0.5),
]

# ── ESMFold pass 1 ────────────────────────────────────────────────────────────
PLDDT_THRESHOLD      = 70.0

# ── ProteinMPNN ───────────────────────────────────────────────────────────────
MPNN_SEQS_PER_STRUCT = 4
MPNN_SAMPLING_TEMP   = 0.1

# ── Protenix ──────────────────────────────────────────────────────────────────
PROTENIX_MODEL       = 'protenix_mini_default_v0.5.0'
MIN_RANKING_SCORE    = 0.5
MIN_EPITOPE_CONTACTS = 1
CONTACT_DIST_A       = 5.0

# ── Final selection ───────────────────────────────────────────────────────────
TOP_N = 10

# ─────────────────────────────────────────────────────────────────────────────
for d in ['esm1_pdbs', 'mpnn_seqs', 'protenix_runs', 'passing']:
    os.makedirs(f'/content/{d}', exist_ok=True)

print(f'TAU_STATE        : {TAU_STATE}')
print(f'Target sequence  : {TARGET_SEQUENCE[:40]}... ({len(TARGET_SEQUENCE)} aa)')
print(f'ESMFold threshold: pLDDT >= {PLDDT_THRESHOLD}')
print(f'MPNN seqs/struct : {MPNN_SEQS_PER_STRUCT}')
print(f'Protenix model   : {PROTENIX_MODEL}')
print(f'Top N            : {TOP_N}')


In [ ]:
import pandas as pd
import os

if os.path.exists('/content/forge_sequences.csv'):
    forge_df = pd.read_csv('/content/forge_sequences.csv')
    print(f'\nTotal Forge sequences loaded: {len(forge_df)}')
    display(forge_df.head())
else:
    print("Error: /content/forge_sequences.csv not found. Check the output of the generation cell for errors.")

## 4. ESMFold pass 1 — fold Forge sequences (backbones for ProteinMPNN)

In [ ]:
print('Loading ESMFold...')
esmfold = esm.pretrained.esmfold_v1().eval().cuda()
print('ESMFold ready.')

def fold_and_plddt(seq, out_path):
    """Fold sequence, write PDB, return mean pLDDT."""
    with torch.no_grad():
        pdb_str = esmfold.infer_pdb(seq)
    with open(out_path, 'w') as f:
        f.write(pdb_str)
    vals = [float(l[60:66]) for l in pdb_str.splitlines() if l.startswith('ATOM')]
    return np.mean(vals) if vals else 0.0

plddt1, pdbs1 = [], []
print(f'ESMFold pass 1: {len(forge_df)} sequences...')
for i, row in forge_df.iterrows():
    out = f"/content/esm1_pdbs/{row['name']}.pdb"
    plddt1.append(fold_and_plddt(row['sequence'], out))
    pdbs1.append(out)
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(forge_df)}')

forge_df['plddt1'] = plddt1
forge_df['pdb1']   = pdbs1

pass1 = forge_df[forge_df['plddt1'] >= PLDDT_THRESHOLD].copy()
print(f'\nPass 1 pLDDT ≥ {PLDDT_THRESHOLD}: {len(pass1)}/{len(forge_df)}')

## 5. ProteinMPNN — redesign sequences on ESMFold backbones (monomer mode)

In [ ]:
import glob as _glob, json as _json

def run_mpnn_monomer(pdb_path, out_dir, n_seqs=MPNN_SEQS_PER_STRUCT,
                     temp=MPNN_SAMPLING_TEMP):
    """Run ProteinMPNN on a binder-only PDB — all residues designed."""
    stem    = os.path.splitext(os.path.basename(pdb_path))[0]
    run_dir = os.path.join(out_dir, stem)
    os.makedirs(run_dir, exist_ok=True)

    jsonl = os.path.join(run_dir, 'input.jsonl')
    with open(jsonl, 'w') as f:
        f.write(_json.dumps({'PDB': pdb_path}) + '\n')

    subprocess.run([
        'python', '/content/ProteinMPNN/protein_mpnn_run.py',
        '--jsonl_path',         jsonl,
        '--out_folder',         run_dir,
        '--num_seq_per_target', str(n_seqs),
        '--sampling_temp',      str(temp),
        '--batch_size',         '1',
    ], capture_output=True, text=True)

    seqs = []
    for fa in _glob.glob(os.path.join(run_dir, 'seqs', '*.fa')):
        with open(fa) as fh:
            lines = fh.readlines()
        for i, line in enumerate(lines):
            if line.startswith('>') and i + 1 < len(lines):
                seqs.append(lines[i + 1].strip())
    return seqs

mpnn_records = []
print(f'ProteinMPNN on {len(pass1)} backbones ({MPNN_SEQS_PER_STRUCT} seqs each)...')
for _, row in pass1.iterrows():
    seqs = run_mpnn_monomer(row['pdb1'], '/content/mpnn_seqs')
    for i, seq in enumerate(seqs):
        mpnn_records.append({
            'name':             f"{row['name']}_mpnn_{i+1:02d}",
            'sequence':         seq,
            'forge_parent':     row['name'],
            'method':           'Forge',
            'target_conformer': row['target_conformer'],
        })

mpnn_df = pd.DataFrame(mpnn_records)
print(f'ProteinMPNN sequences ready: {len(mpnn_df)}')

# Free GPU before loading Protenix
del esmfold
torch.cuda.empty_cache()
print('ESMFold unloaded — GPU freed for Protenix.')

## 6. Protenix — predict binder + K18 complex (replaces ESMFold pass 2)

In [ ]:
# ── Protenix helper functions ─────────────────────────────────────────────────

def run_protenix(name, binder_seq, target_seq, out_base, model=PROTENIX_MODEL):
    """
    Predict a binder+target complex with Protenix.
    Returns (ranking_score, iptm, cif_path) or (None, None, None) on failure.
    Chain A = target (K18), Chain B = binder.
    """
    run_dir  = os.path.join(out_base, name)
    os.makedirs(run_dir, exist_ok=True)

    input_data = [{
        'name': name,
        'sequences': [
            {'proteinChain': {'sequence': target_seq, 'count': 1}},
            {'proteinChain': {'sequence': binder_seq, 'count': 1}},
        ]
    }]
    input_json = os.path.join(run_dir, 'input.json')
    with open(input_json, 'w') as f:
        _json.dump(input_data, f)

    result = subprocess.run([
        'protenix', 'pred',
        '-i', input_json,
        '-o', run_dir,
        '-n', model,
        '-s', '101',
        '-e', '1',
        '--use_msa', 'false',
    ], capture_output=True, text=True, timeout=600)

    # Find confidence JSON
    conf_files = _glob.glob(os.path.join(run_dir, '**', '*confidence*.json'), recursive=True)
    if not conf_files:
        return None, None, None

    with open(conf_files[0]) as f:
        conf = _json.load(f)
    iptm    = conf.get('iptm', 0.0)
    ptm     = conf.get('ptm',  0.0)
    ranking = 0.8 * iptm + 0.2 * ptm

    cif_files = _glob.glob(os.path.join(run_dir, '**', '*.cif'), recursive=True)
    cif_path  = cif_files[0] if cif_files else None

    return ranking, iptm, cif_path


print('Protenix helper defined.')

In [ ]:
# ── Epitope contact analysis from CIF ────────────────────────────────────────
from Bio.PDB import MMCIFParser
import warnings
from Bio.PDB.PDBExceptions import PDBConstructionWarning

def epitope_contacts_from_cif(cif_path, epitope_ranges=EPITOPE_RANGES,
                               cutoff=CONTACT_DIST_A):
    """
    Count binder residues within `cutoff` Å of any epitope residue.
    Chain A = target (K18), Chain B = binder.
    Returns total_contacts and per-epitope breakdown dict.
    """
    if cif_path is None or not os.path.exists(cif_path):
        return 0, {}

    with warnings.catch_warnings():
        warnings.simplefilter('ignore', PDBConstructionWarning)
        parser = MMCIFParser(QUIET=True)
        structure = parser.get_structure('complex', cif_path)

    model   = structure[0]
    chains  = list(model.get_chains())
    if len(chains) < 2:
        return 0, {}

    chain_A = chains[0]   # target K18
    chain_B = chains[1]   # binder

    target_residues = list(chain_A.get_residues())
    binder_atoms    = list(chain_B.get_atoms())

    breakdown  = {}
    total_set  = set()

    for start, end, label, weight in epitope_ranges:
        epi_atoms = []
        for res in target_residues[start:end]:
            epi_atoms.extend(res.get_atoms())

        contacts = set()
        for e_atom in epi_atoms:
            for b_atom in binder_atoms:
                if (e_atom.coord - b_atom.coord).__abs__() < cutoff:
                    contacts.add(b_atom.get_parent().get_id()[1])

        breakdown[label] = len(contacts)
        total_set |= contacts

    return len(total_set), breakdown


print('Epitope contact function defined.')

In [ ]:
# ── Run Protenix on all MPNN sequences ───────────────────────────────────────
protenix_records = []
total = len(mpnn_df)
print(f'Running Protenix on {total} sequences...')

for i, row in mpnn_df.iterrows():
    ranking, iptm, cif_path = run_protenix(
        name       = row['name'],
        binder_seq = row['sequence'],
        target_seq = TARGET_SEQUENCE,
        out_base   = '/content/protenix_runs',
    )

    if ranking is None:
        contacts, breakdown = 0, {}
        ranking = iptm = 0.0
    else:
        contacts, breakdown = epitope_contacts_from_cif(cif_path)

    protenix_records.append({
        **row.to_dict(),
        'ranking_score':     ranking,
        'iptm':              iptm,
        'epitope_contacts':  contacts,
        'epitope_breakdown': str(breakdown),
        'cif_path':          cif_path,
    })

    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{total}  |  last ranking={ranking:.3f}  contacts={contacts}')

results_df = pd.DataFrame(protenix_records)
print(f'\nProtenix done. Median ranking score: {results_df["ranking_score"].median():.3f}')

## 7. Filter — ranking score + epitope contacts

In [ ]:
mask = (
    (results_df['ranking_score']    >= MIN_RANKING_SCORE) &
    (results_df['epitope_contacts'] >= MIN_EPITOPE_CONTACTS)
)
passing = results_df[mask].copy()

print(f'Ranking ≥ {MIN_RANKING_SCORE} AND contacts ≥ {MIN_EPITOPE_CONTACTS}:')
print(f'  Passing: {len(passing)} / {len(results_df)}')
if len(passing):
    print(f'  Ranking  : {passing["ranking_score"].min():.3f} – {passing["ranking_score"].max():.3f}')
    print(f'  ipTM     : {passing["iptm"].min():.3f} – {passing["iptm"].max():.3f}')
    print(f'  Contacts : {passing["epitope_contacts"].min()} – {passing["epitope_contacts"].max()}')
else:
    print('  No passing sequences — consider relaxing thresholds.')

## 8. QC plots

In [ ]:
import matplotlib.pyplot as plt, shutil

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# pLDDT pass 1
ax = axes[0, 0]
ax.hist(forge_df['plddt1'], bins=20, color='steelblue', edgecolor='white')
ax.axvline(PLDDT_THRESHOLD, color='tomato', ls='--', lw=2, label=f'≥{PLDDT_THRESHOLD}')
ax.set_xlabel('pLDDT'); ax.set_title('ESMFold pass 1 (Forge seqs)'); ax.legend(fontsize=8)

# Ranking score distribution
ax = axes[0, 1]
ax.hist(results_df['ranking_score'], bins=25, color='darkorange', edgecolor='white')
ax.axvline(MIN_RANKING_SCORE, color='tomato', ls='--', lw=2, label=f'≥{MIN_RANKING_SCORE}')
ax.set_xlabel('Ranking score (0.8·ipTM + 0.2·pTM)')
ax.set_title('Protenix ranking scores'); ax.legend(fontsize=8)

# ipTM distribution
ax = axes[0, 2]
ax.hist(results_df['iptm'], bins=25, color='seagreen', edgecolor='white')
ax.set_xlabel('ipTM'); ax.set_title('Protenix ipTM')

# Epitope contacts
ax = axes[1, 0]
ax.hist(results_df['epitope_contacts'], bins=range(0, 30), color='mediumpurple', edgecolor='white')
ax.axvline(MIN_EPITOPE_CONTACTS - 0.5, color='tomato', ls='--', lw=2)
ax.set_xlabel('Epitope contacts (# binder residues)'); ax.set_title('Epitope contacts')

# Ranking vs contacts scatter
ax = axes[1, 1]
ax.scatter(results_df['epitope_contacts'], results_df['ranking_score'],
           alpha=0.4, s=15, color='slategray')
if len(passing):
    ax.scatter(passing['epitope_contacts'], passing['ranking_score'],
               alpha=0.8, s=25, color='tomato', label='Passing')
ax.axhline(MIN_RANKING_SCORE, color='tomato', ls='--', lw=1)
ax.axvline(MIN_EPITOPE_CONTACTS - 0.5, color='tomato', ls='--', lw=1)
ax.set_xlabel('Epitope contacts'); ax.set_ylabel('Ranking score')
ax.set_title('Ranking vs contacts'); ax.legend(fontsize=8)

# Passing binder length distribution
ax = axes[1, 2]
if len(passing):
    ax.hist(passing['sequence'].str.len(), bins=15, color='steelblue', edgecolor='white')
else:
    ax.text(0.5, 0.5, 'No passing sequences', ha='center', va='center', transform=ax.transAxes)
ax.set_xlabel('Binder length (aa)'); ax.set_title(f'Passing lengths (n={len(passing)})')

plt.suptitle('Forge → ProteinMPNN → Protenix pipeline QC', fontsize=13)
plt.tight_layout()
plt.savefig('/content/forge_pipeline_qc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: forge_pipeline_qc.png')

## 9. Save — CSV + CIF files

In [ ]:
# Copy passing CIFs
for _, row in passing.iterrows():
    if row['cif_path'] and os.path.exists(row['cif_path']):
        shutil.copy2(row['cif_path'], f"/content/passing/{row['name']}.cif")

# Add tau_state column so NB03 can identify the condition
passing = passing.copy()
passing['tau_state'] = TAU_STATE

out_cols = ['name', 'sequence', 'iptm', 'ranking_score', 'method',
            'target_conformer', 'forge_parent', 'epitope_contacts',
            'epitope_breakdown', 'tau_state']
out_cols = [c for c in out_cols if c in passing.columns]

out_csv = f'/content/forge_{TAU_STATE}_top{TOP_N}.csv'
passing[out_cols].to_csv(out_csv, index=False)

if '_in_drive' in dir() and _in_drive:
    import shutil as _sh
    _sh.copy2(out_csv, os.path.join(NB02C_OUT, os.path.basename(out_csv)))
    print(f'Saved to Drive: {NB02C_OUT}')

print(f'Saved: {out_csv}  ({len(passing)} rows)')
n_cif = len(os.listdir('/content/passing'))
print(f'CIF files: {n_cif}')
passing[out_cols].head(10)


## 10. Download

In [ ]:
import zipfile

zip_name = f'/content/forge_{TAU_STATE}_designs.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(out_csv, os.path.basename(out_csv))
    zf.write('/content/forge_pipeline_qc.png', f'forge_{TAU_STATE}_qc.png')
    for fn in os.listdir('/content/passing'):
        zf.write(f'/content/passing/{fn}', f'structures/{fn}')

import shutil as _shutil
if '_in_drive' in dir() and _in_drive:
    _shutil.copy2(zip_name, os.path.join(NB02C_OUT, os.path.basename(zip_name)))
    _shutil.copy2(out_csv, os.path.join(NB02C_OUT, os.path.basename(out_csv)))
    print(f'Saved to Drive: {NB02C_OUT}')

zip_kb = os.path.getsize(zip_name) // 1024
zip_base = os.path.basename(zip_name)
print(f'Zip ready: {zip_base}  ({zip_kb} KB)')
print(f'Contents:')
print(f'  forge_{TAU_STATE}_top{TOP_N}.csv  <- upload to NB03')
print( '  structures/*.cif')
print(f'\nNext: change TAU_STATE to oligomer / fibril and re-run.')
print( 'NB03 expects forge_monomer, forge_oligomer, forge_fibril + 3 BindCraft CSVs.')

try:
    from google.colab import files as _colab_files
    _colab_files.download(zip_name)
except ImportError:
    print(f'Not in Colab - file at: {zip_name}')
